# 06 - Machine Learning Model Training

This notebook trains three classifiers using the preprocessing pipeline created in Notebook 05.

In [1]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

project_root = Path("..").resolve()
preprocessing_artifact = joblib.load(project_root / "models" / "churn_preprocessing.joblib")
preprocessor = preprocessing_artifact["pipeline"]
split_artifact = joblib.load(project_root / "models" / "churn_train_test.joblib")

X_train = split_artifact["X_train"]
X_test = split_artifact["X_test"]
y_train = split_artifact["y_train"]
y_test = split_artifact["y_test"]
print("Loaded training rows:", len(X_train))
print("Loaded test rows:", len(X_test))

Loaded training rows: 5634
Loaded test rows: 1409


## Class Imbalance Strategy

The target is imbalanced, with fewer churned customers. `class_weight='balanced'` is used for Logistic Regression, Decision Tree, and Random Forest to give minority-class errors greater influence during training.

In [2]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

trained_pipelines = {}
comparison_rows = []

for model_name, classifier in models.items():
    complete_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    complete_pipeline.fit(X_train, y_train)

    predictions = complete_pipeline.predict(X_test)
    probabilities = complete_pipeline.predict_proba(X_test)[:, 1]

    trained_pipelines[model_name] = complete_pipeline
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

model_comparison = pd.DataFrame(comparison_rows).set_index("Model")
model_comparison

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841639
Decision Tree,0.731725,0.494565,0.486631,0.490566,0.653696
Random Forest,0.783534,0.621053,0.473262,0.537178,0.822690


## Algorithm Notes

- **Logistic Regression:** An interpretable linear baseline that estimates log-odds.
- **Decision Tree:** Learns explicit rules and can capture non-linear patterns, but may overfit.
- **Random Forest:** Combines many decision trees to reduce variance and improve generalization.

In [3]:
joblib.dump(trained_pipelines, project_root / "models" / "churn_model_pipelines.joblib")
model_comparison.to_csv(project_root / "models" / "model_comparison.csv")
print("Saved model pipelines and model comparison table.")

Saved model pipelines and model comparison table.
